# ICT Market Structure — Demo
> `strategies/market_structure.py` — Swing detection · BOS · CHoCH · Order Blocks

In [112]:
import sys
sys.path.insert(0, r"F:\Claude Projects\quant_ict_trader")

In [113]:
import importlib
import strategies.market_structure as ms_module
importlib.reload(ms_module)
from strategies.market_structure import MarketStructure, analyse

In [114]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import yfinance as yf
import pandas as pd
from strategies.market_structure import MarketStructure, analyse

## 1 · Load EURUSD 1H data

In [115]:
raw = yf.download('EURUSD=X', period='60d', interval='1h', auto_adjust=True)

# yfinance returns MultiIndex columns — flatten to single level
df = raw.copy()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0).str.lower()
else:
    df.columns = df.columns.str.lower()
df.index = pd.to_datetime(df.index)
df = df.dropna()

print(f'Rows: {len(df):,}  |  {df.index[0]} → {df.index[-1]}')
df.tail(3)

[*********************100%***********************]  1 of 1 completed

Rows: 1,408  |  2026-02-04 00:00:00+00:00 → 2026-04-28 07:00:00+00:00


Price,close,high,low,open,volume
Datetime,,,,,
2026-04-28 05:00:00+00:00,1.170960,1.171646,1.170960,1.171646,0
2026-04-28 06:00:00+00:00,1.169180,1.171097,1.169180,1.170823,0
2026-04-28 07:00:00+00:00,1.169727,1.170412,1.169044,1.169044,0


## 2 · Run analysis

In [116]:
ms = MarketStructure(df, swing_lookback=5)

print(f'Swing Highs : {len(ms.swing_highs)}')
print(f'Swing Lows  : {len(ms.swing_lows)}')
print(f'BOS/CHoCH   : {len(ms.structure_events)}')
print(f'Order Blocks: {len(ms.order_blocks)}')

Swing Highs : 110
Swing Lows  : 105
BOS/CHoCH   : 55
Order Blocks: 55


## 3 · Interactive chart (last 300 candles)

In [117]:
fig = ms.plot(
    last_n=300,
    title='EURUSD 1H — ICT Market Structure',
    height=820,
    max_events=10,           # show only 10 most recent BOS/CHoCH
    show_mitigated_ob=False, # hide already-visited OBs
)
fig.show()

## 4 · Events summary table

In [118]:
ms.summary().tail(15)

,timestamp,event,price,broken_level
40,2026-04-06 06:00:00+00:00,CHoCH_bull,1.153669,1.153270
41,2026-04-07 01:00:00+00:00,CHoCH_bear,1.153270,1.153802
42,2026-04-07 17:00:00+00:00,CHoCH_bull,1.158078,1.157809
43,2026-04-08 12:00:00+00:00,BOS_bull,1.171783,1.171097
44,2026-04-10 06:00:00+00:00,CHoCH_bear,1.168770,1.168907
45,2026-04-10 13:00:00+00:00,CHoCH_bull,1.173020,1.172608
46,2026-04-13 13:00:00+00:00,BOS_bull,1.170686,1.170275
47,2026-04-16 02:00:00+00:00,BOS_bull,1.182033,1.181056
48,2026-04-17 08:00:00+00:00,BOS_bull,1.179802,1.179106
49,2026-04-17 19:00:00+00:00,CHoCH_bear,1.177163,1.177718


## 5 · Order blocks table

In [119]:
ms.order_blocks_df()

,timestamp,kind,high,low,open,close,mitigated,mitigated_at
0,2026-02-05 00:00:00+00:00,bearish,1.180777,1.180359,1.180498,1.180638,True,2026-02-05 01:00:00+00:00
1,2026-02-05 19:00:00+00:00,bearish,1.180080,1.179663,1.179663,1.179802,True,2026-02-05 20:00:00+00:00
2,2026-02-06 11:00:00+00:00,bullish,1.179663,1.178967,1.179523,1.179384,True,2026-02-06 12:00:00+00:00
3,2026-02-06 21:00:00+00:00,bullish,1.182732,1.181754,1.182592,1.182173,True,2026-02-06 22:00:00+00:00
4,2026-02-10 09:00:00+00:00,bearish,1.192321,1.191185,1.191327,1.191753,True,2026-02-10 10:00:00+00:00
5,2026-02-13 02:00:00+00:00,bearish,1.187507,1.186662,1.186944,1.187225,True,2026-02-13 03:00:00+00:00
6,2026-02-16 10:00:00+00:00,bearish,1.187085,1.186662,1.186803,1.187085,True,2026-02-16 11:00:00+00:00
7,2026-02-17 19:00:00+00:00,bullish,1.185396,1.184694,1.185255,1.185115,True,2026-02-17 20:00:00+00:00
8,2026-02-18 15:00:00+00:00,bearish,1.182872,1.181893,1.181893,1.182592,True,2026-02-18 16:00:00+00:00
9,2026-02-19 07:00:00+00:00,bearish,1.180638,1.179663,1.179941,1.180498,True,2026-02-19 08:00:00+00:00


## 6 · 4H chart
Higher-timeframe context for confluence.

In [120]:
raw_4h = yf.download('EURUSD=X', period='120d', interval='4h', auto_adjust=True)
df_4h = raw_4h.copy()
if isinstance(df_4h.columns, pd.MultiIndex):
    df_4h.columns = df_4h.columns.get_level_values(0).str.lower()
else:
    df_4h.columns = df_4h.columns.str.lower()
df_4h.index = pd.to_datetime(df_4h.index)
df_4h = df_4h.dropna()

ms_4h = MarketStructure(df_4h, swing_lookback=5)
ms_4h.plot(
    last_n=200,
    title='EURUSD 4H — ICT Market Structure',
    height=820,
).show()

[*********************100%***********************]  1 of 1 completed


In [121]:
import importlib
import strategies.fvg as fvg_module
importlib.reload(fvg_module)
from strategies.fvg import FairValueGap

fvg = FairValueGap(df, min_gap_pips=2.0)
print(f"Total FVGs : {len(fvg.fvgs)}")
print(f"Unfilled   : {len(fvg.unfilled())}")
fvg.plot(last_n=300, title="EURUSD 1H — Fair Value Gaps").show()

Total FVGs : 259
Unfilled   : 13


In [122]:
import importlib
import strategies.liquidity as liq_module
importlib.reload(liq_module)
from strategies.liquidity import Liquidity

liq = Liquidity(df, swing_lookback=5, equal_threshold=3.0)
print(f"Equal Highs : {len(liq.equal_highs)}")
print(f"Equal Lows  : {len(liq.equal_lows)}")
print(f"Grabs       : {len(liq.grabs)}")
print(f"Active Buy-side  : {len(liq.active_buyside())}")
print(f"Active Sell-side : {len(liq.active_sellside())}")
liq.plot(last_n=300, title="EURUSD 1H — Liquidity").show()

Equal Highs : 28
Equal Lows  : 22
Grabs       : 1158
Active Buy-side  : 9
Active Sell-side : 6


In [123]:
import importlib
import strategies.entry_model as em_module
importlib.reload(em_module)
from strategies.entry_model import EntryModel

# Download 4H data
raw_4h = yf.download('EURUSD=X', period='120d', interval='4h', auto_adjust=True)
df_4h = raw_4h.copy()
if isinstance(df_4h.columns, pd.MultiIndex):
    df_4h.columns = df_4h.columns.get_level_values(0).str.lower()
df_4h.index = pd.to_datetime(df_4h.index)
df_4h = df_4h.dropna()

# Run model
model = EntryModel(
    df_high=df_4h,
    df_low=df,
    instrument="EURUSD",
    account_balance=10000,
    risk_pct=0.01,
    daily_loss_limit=0.03,
    max_drawdown=0.10,
    tp1_rr=1.0,
    tp2_rr=2.0,
    sl_buffer_pips=3.0,
    min_rr=1.5,
)

print(f"Signals found: {len(model.signals)}")
if model.latest_signal():
    print(model.latest_signal())

model.plot(last_n=300).show()

fig = model.plot(last_n=100)
fig.write_html("chart.html")
print("Open chart.html in your browser")
print(model.latest_signal())
print(f"\nTotal signals: {len(model.signals)}")
print(model.signals_df())

# Check what's being detected
print("HTF Trend:", model._get_htf_trend())
print("LTF Trend:", model._get_ltf_trend())
print()
print("Unfilled FVGs:", len(model.fvg.unfilled()))
for f in model.fvg.unfilled():
    size = (f.top - f.bottom) / 0.0001
    print(f"  {f.kind} FVG | size: {size:.1f} pips | top: {f.top:.5f} | bottom: {f.bottom:.5f}")
print()
print("Active OBs:", len([o for o in model.ms_low.order_blocks if not o.mitigated]))
for ob in [o for o in model.ms_low.order_blocks if not o.mitigated]:
    sl_dist = abs(ob.high - ob.low) / 0.0001
    print(f"  {ob.kind} OB | size: {sl_dist:.1f} pips | high: {ob.high:.5f} | low: {ob.low:.5f}")

current = model.df_low["close"].iloc[-1]
print(f"Current price: {current:.5f}")
print(f"\nLast grab: {model.liq.grabs[-1] if model.liq.grabs else 'None'}")
print(f"Active buyside (EQH): {len(model.liq.active_buyside())}")
for eq in model.liq.active_buyside():
    print(f"  EQH at {eq.price:.5f}")

[*********************100%***********************]  1 of 1 completed


Signals found: 7

  SHORT EURUSD
  Entry  : 1.17130
  SL     : 1.17608  (47.8 pips)
  TP1    : 1.16653  (RR 1.0:1)
  TP2    : 1.16175  (RR 2.0:1)
  Size   : 0.30 lots
  Risk   : $100.00 (1.0%)
  Trend  : BOS_bear on HTF
  Zone   : Bearish FVG
  Confirm: EQH confluence at 1.17578


Open chart.html in your browser

  SHORT EURUSD
  Entry  : 1.17130
  SL     : 1.17608  (47.8 pips)
  TP1    : 1.16653  (RR 1.0:1)
  TP2    : 1.16175  (RR 2.0:1)
  Size   : 0.30 lots
  Risk   : $100.00 (1.0%)
  Trend  : BOS_bear on HTF
  Zone   : Bearish FVG
  Confirm: EQH confluence at 1.17578

Total signals: 7
                  timestamp direction    entry       sl      tp1      tp2  \
0 2026-04-28 07:00:00+00:00     short  1.19161  1.17608  1.17608  1.16754   
1 2026-04-28 07:00:00+00:00     short  1.19090  1.17608  1.17608  1.16754   
2 2026-04-28 07:00:00+00:00     short  1.18231  1.17608  1.17608  1.16754   
3 2026-04-28 07:00:00+00:00     short  1.18120  1.17608  1.17608  1.16754   
4 2026-04-28 07:00:00+00:00     short  1.18022  1.17608  1.17608  1.17194   
5 2026-04-28 07:00:00+00:00     short  1.17385  1.17608  1.17161  1.16754   
6 2026-04-28 07:00:00+00:00     short  1.17130  1.17608  1.16653  1.16175   

   sl_pips  rr_tp1  rr_tp2  size_lots  risk_$         zone  \
0    155

In [124]:
import importlib, strategies.entry_model as em
importlib.reload(em)
from strategies.entry_model import EntryModel

model = EntryModel(df_high=df_4h, df_low=df, instrument="EURUSD",
    account_balance=10000, risk_pct=0.01, daily_loss_limit=0.03,
    max_drawdown=0.10, tp1_rr=1.0, tp2_rr=2.0, sl_buffer_pips=3.0, min_rr=1.5)

print(f"Signals: {len(model.signals)}")
print(model._find_entry_zones("short")[:3])

Signals: 7
[(np.float64(1.1916111707687378), 'Bearish FVG', np.float64(1.1760789850234985)), (np.float64(1.1909014582633972), 'Bearish FVG', np.float64(1.1760789850234985)), (np.float64(1.1823129057884216), 'Bearish FVG', np.float64(1.1760789850234985))]


In [127]:
from utils.signal_viewer import plot_all_signals

plot_all_signals(
    df=df,
    signals=model.signals,
    candles_before=60,
    candles_after=20,
    save_html=r"F:\Claude Projects\quant_ict_trader\signals.html"
)

UnicodeEncodeError: 'charmap' codec can't encode character '\u95f0' in position 4786571: character maps to <undefined>